# Document AI

**Module:** 15 — VLMs & Multimodal

Turn PDFs, scans, and forms into reliable structured data with layout-aware multimodal pipelines.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Map ingest → classify → extract → validate → HITL → export
- Explain layout challenges: tables, columns, stamps, handwriting
- Compare layout-aware strategies (OCR+geometry vs VLM)
- Prototype schema extraction with confidence routing


## Document AI Pipeline

### Definition
Document AI converts document images/PDFs into validated business objects with auditability.

### Why it matters
Manual entry doesn't scale; brittle regex scrapers die on new templates.

### How it works
Ingest → classify → rasterize/select → layout/OCR/VLM → extract → validate → HITL → export. Persist provenance.

### Intuition
An assembly line: each station reduces ambiguity.

### Pitfalls
- One prompt for all doc types
- No schema validation
- Losing page numbers

### When to use
Invoices, KYC, claims, contracts, medical forms.


### Pipeline diagram

```mermaid
flowchart TD
  A[Ingest] --> B[MIME / malware scan]
  B --> C[Classify]
  C --> D[Rasterize / page select]
  D --> E[Layout + OCR and/or VLM]
  E --> F[JSON fields]
  F --> G{Validate}
  G -->|fail| H[HITL]
  G -->|pass| I[Export]
  H --> I
```

| Stage | Question | Artifact |
|-------|----------|----------|
| Classify | Template family? | `doc_type` |
| Analyze | Where is content? | blocks/tables |
| Extract | What values? | JSON |
| Validate | Trustworthy? | errors[], conf |


In [ ]:
# Demo 1: ingest + classify
from dataclasses import dataclass, field
from typing import Any
import hashlib

@dataclass
class DocJob:
    filename: str; mime: str; bytes_preview: bytes
    doc_type: str | None = None
    fields: dict[str, Any] = field(default_factory=dict)
    errors: list[str] = field(default_factory=list)
    @property
    def content_hash(self): return hashlib.sha256(self.bytes_preview).hexdigest()[:12]

def classify(job: DocJob) -> DocJob:
    n = job.filename.lower()
    job.doc_type = "invoice" if "invoice" in n or "inv-" in n else "id" if "passport" in n or "id_" in n else "unknown"
    return job
print(classify(DocJob("Acme_Invoice_1042.pdf","application/pdf",b"%PDF")).doc_type)


In [ ]:
# Demo 2: invoice schema validation
from datetime import datetime
SCHEMA = {"vendor": str, "invoice_id": str, "total": float, "currency": str, "date": str}

def validate_invoice(f: dict) -> list[str]:
    errs = []
    for k, typ in SCHEMA.items():
        if k not in f: errs.append(f"missing:{k}"); continue
        if not isinstance(f[k], typ): errs.append(f"type:{k}")
    if f.get("total", 0) < 0: errs.append("total_negative")
    try: datetime.strptime(f.get("date",""), "%Y-%m-%d")
    except Exception: errs.append("date_format")
    if f.get("currency") not in {"USD","EUR","INR","GBP"}: errs.append("currency")
    return errs

print(validate_invoice({"vendor":"Acme","invoice_id":"1","total":19.99,"currency":"USD","date":"2026-07-01"}))
print(validate_invoice({"vendor":"Acme","total":-1,"currency":"BTC","date":"07/01/26"}))


## Challenges Unique to Documents

### Definition
Documents mix print, handwriting, tables, stamps, logos, multi-column order — not natural photos.

### Why it matters
Reading-order errors flip amounts; table misparses corrupt line items.

### How it works
Detect rotation first; special-case tables; consider hybrid OCR+VLM.

### Intuition
Humans bring layout priors; systems must be given or learn them.

### Pitfalls
- Trusting PDF text layer that mismatches the scan
- Assuming single-column LTR

### When to use
Any non-trivial scanned or born-digital document.


### Layout-aware strategies

| Strategy | Strength | Weakness |
|----------|----------|----------|
| PDF text layer | Cheap/exact | Useless on scans |
| OCR + boxes | Auditable geometry | Brittle templates |
| LayoutLMs | Strong forms | Domain care |
| VLM page QA | Flexible | Cost/hallucination |
| Hybrid reconcile | Best reliability | More engineering |

**Hybrid:** OCR proposes characters; VLM assigns fields; validators check math.


In [ ]:
# Demo 3: reading-order sort
words = [
    {"text":"Total","x":0.7,"y":0.8},{"text":"Invoice","x":0.1,"y":0.05},
    {"text":"1042","x":0.25,"y":0.05},{"text":"$19.99","x":0.82,"y":0.8},
    {"text":"Acme LLC","x":0.1,"y":0.12},
]
def reading_order(words, tol=0.03):
    return [w["text"] for w in sorted(words, key=lambda w: (round(w["y"]/tol), w["x"]))]
print(reading_order(words))


In [ ]:
# Demo 4: confidence routing
def route(fields, conf, thr=0.85):
    low = [k for k,v in conf.items() if v < thr]
    math_ok = abs(fields.get("subtotal",0)+fields.get("tax",0)-fields.get("total",0)) < 0.01
    return ("HITL: " + str(low) + f" math={math_ok}") if low or not math_ok else "auto_export"
print(route({"subtotal":10,"tax":2,"total":12}, {"vendor":0.95,"total":0.99,"invoice_id":0.7}))
print(route({"subtotal":10,"tax":2,"total":13}, {"vendor":0.95,"total":0.99,"invoice_id":0.95}))


In [ ]:
# Demo 5: VLM extract request shape
import json
req = {
    "model": "gpt-4o",
    "messages": [{"role":"user","content":[
        {"type":"text","text":"Extract JSON keys: vendor, invoice_id, date, currency, total, line_items[]. null if not visible."},
        {"type":"image_url","image_url":{"url":"data:image/png;base64,YOUR_PAGE_BASE64"}},
    ]}],
    "response_format": {"type": "json_object"},
}
print(json.dumps(req, indent=2)[:380])
print("OPENAI_API_KEY=YOUR_OPENAI_API_KEY")


In [ ]:
# Demo 6: blank-page skip heuristic + provenance
def mean_ink(gray_bytes_sample: list[int]) -> float:
    return sum(gray_bytes_sample)/len(gray_bytes_sample)/255

def select_pages(pages):
    out = []
    for p in pages:
        if mean_ink(p["sample"]) > 0.98:  # nearly white
            continue
        out.append({"page": p["n"], "provenance": f"page-{p['n']}"})
    return out
print(select_pages([{"n":1,"sample":[250]*20},{"n":2,"sample":[40]*20}]))


### Error taxonomy

| Error | Example | Guard |
|-------|---------|-------|
| Field swap | Tax ↔ total | Schema + math |
| Currency | BTC / missing | Allow-list |
| Date locale | 07/01/26 | Normalize ISO |
| Vendor halluc | Plausible name | OCR overlap check |
| Line drift | Wrong SKU row | Table structure / HITL |


### Checklist — DocAI go-live

- [ ] Per-doc-type schema
- [ ] Provenance page/bbox stored
- [ ] HITL queue for low conf
- [ ] Model+prompt version logged
- [ ] Reprocess tool for template drift


### Try it yourself — Document pipeline

1. Add id-card schema distinct from invoice.
2. Reconciler preferring OCR digits when VLM total disagrees >1%.
3. Emit `{field, page, bbox}` provenance for each value.

**Stretch:** Simulate 3-page jobs and skip blanks via mean_ink.


### Try it yourself — Tables

1. Design a JSON schema for line_items with qty/unit/price.
2. List 4 ways table extraction fails on merged cells.


## Knowledge Check

**Q1.** Why classify before extract?

<details><summary>Answer</summary>

Schemas, prompts, and validators differ by document family; one mega-prompt underperforms and fails audits.

</details>

**Q2.** What does hybrid OCR+VLM buy you?

<details><summary>Answer</summary>

Character evidence + flexible field assignment + disagreement signals for HITL.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `rasterize` | Render PDF page to pixels |
| `reading order` | Human/model read sequence |
| `provenance` | Where a field came from |
| `born-digital` | PDF with real text layer |
| `template drift` | Layout changes breaking extractors |
| `HITL` | Human-in-the-loop |


## Key Takeaways

- DocAI is classify → extract → validate → HITL
- Layout/reading order are first-class
- Hybrids beat single-technique systems
- Schema + confidence routing make automation shippable


## Production Incident Patterns — document AI

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "document AI",
}))


## Mini Case Study — document AI

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("document AI", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — document AI ops

1. Write a one-page runbook section for on-call when document AI critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
